# Assessing Gender Bias in LLM Embeddings

### Introduction

The goal of this project is to apply established research techniques for measuring bias in contextual embedding-based models like BERT to state-of-the art large language models (LLMs). 

Embeddings are semantic token representations central to how modern language models interpret their inputs, so understanding how gendered terms are embedded with relation to terms like professions can illustrate latent social biases codified in the model(s). As embeddings have grown in complexity, techniques used to analyze the semantic relations they capture have had to adapt.

### Approach
Bartl et al (2020) used a masked language modeling (MLM) task and log odds scoring based on the word embedding association test (WEAT) from previous literature to measure the relative likelihood of target words given the presence of certain attribute words. 

To that end, they also created the Bias Evaluation Corpus with Professions (BEC-Pro), which contains MLM templates with gendered target words and professions and is annotated with U.S. Department of Labor statistics regarding gender breakdowns.While Bartl et al focus on binary gender representations, they recognize the representational harm of not including non-binary gender markers in their research.

This project aims to take a subset of the BEC-Pro corpus (the entries with "He" and "She" as target words), expand it to include the non-binary pronoun "Ze", and use prompting to re-create their evaluation of bias in BERT, along with the following LLMs:
* GPT 4.1 Nano
* GPT 5.4
* Llama 3.1 8B Instruct (quantized)

<b>Masked vs. Causal Language Modeling</b>

Masked language models like BERT are trained to correctly predict a randomly removed word from a sentence.

Example Input:

`The quick, brown [MASK] jumped over the lazy dog.`

Output:

`fox`

Causal language models like GPT and Llama are trained to predict the next token in a sequence of tokens. However, we can structure prompts for these models so that the predicted token answers a masked language modeling task.

Example Prompt:

```Please complete this MLM task by choosing the best word(s) to replace the [MASK] token(s) in the sentence below. The first masked word should be a pronoun. Return only the masked word(s), separated by spaces.```

 ```The [MASK], brown fox jumped over the lazy dog.```

Output:

`quick`




### Setup

In [1]:
import configparser

import data_utils
import lm_api
import viz

c:\Users\Godly\Documents\Github\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0504 13:50:51.910000 10024 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
config = configparser.ConfigParser()
config.read('config.ini')

['config.ini']

In [3]:
bec_df = data_utils.load_bec_data(config['DATA']['bec_path'])

In [4]:
non_binary_bec_df = data_utils.synthesize_bec_data(bec_df.loc[bec_df['Person'] == 'She'] , target_word='Ze', target_gender='Non-binary')

In [5]:
openai_key = config['CONNECTION']['open_ai_key']
xai_key = config['CONNECTION']['xai_key']
hf_key = config['CONNECTION']['hf_key']
prompt = config['CONNECTION']['prompt_intro']

In [6]:
bert_client = lm_api.BertClient(model='bert-base-uncased')

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 9374.30it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
open_ai_gpt4_1_nano_client = lm_api.OpenAiClient(openai_key, prompt, "gpt-4.1-nano-2025-04-14")
open_ai_gpt5_4_client = lm_api.OpenAiClient(openai_key, prompt, "gpt-5.4-2026-03-05")

In [8]:
llama_3_1_8b_client = lm_api.LlamaClient(model="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4", key=hf_key)

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0+6ce1e7b
Transformers : 5.7.0
Torch        : 2.11.0+cu130
Triton       : 3.6.0.post26


INFO  ExLlamaV2 AWQ: compiling torch.ops JIT extension in `C:\Users\Godly\.cache\gptqmodel\torch_extensions\exllamav2_awq\cee329e012c52736`.


...

INFO  Kernel: Auto-selection: adding candidate `AwqGEMMTritonLinear`           


INFO  Kernel: selected -> `AwqGEMMTritonLinear`.                               


Loading weights: 100%|██████████| 739/739 [00:02<00:00, 342.23it/s]
INFO:httpx:HTTP Request: HEAD https://huggingface.co/hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4/db1f81ad4b8c7e39777509fac66c652eb0a52f91/generation_config.json "HTTP/1.1 200 OK"


INFO  gc.collect() reclaimed 2373 objects in 0.234s                            


INFO:httpx:HTTP Request: HEAD https://huggingface.co/hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4/db1f81ad4b8c7e39777509fac66c652eb0a52f91/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4/db1f81ad4b8c7e39777509fac66c652eb0a52f91/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.c

### Execute Predictions
Requires local model and API configuration, un-comment to run

In [9]:
# bert_client.add_scores_to_df(bec_df, 'bert_score')

In [10]:
# bert_client.add_scores_to_df(non_binary_bec_df, 'bert_score')

In [11]:
# open_ai_gpt4_1_nano_client.add_scores_to_df(bec_df, 'gpt4_1_nano_score')

In [12]:
# open_ai_gpt4_1_nano_client.add_scores_to_df(non_binary_bec_df, 'gpt4_1_nano_score')

In [13]:
# open_ai_gpt5_4_client.add_scores_to_df(bec_df, 'gpt5_4_score')

In [14]:
# open_ai_gpt5_4_client.add_scores_to_df(non_binary_bec_df, 'gpt5_4_score')

In [15]:
# llama_3_1_8b_client.add_scores_to_df(bec_df, 'llama_3_1_8b_score')

In [16]:
# llama_3_1_8b_client.add_scores_to_df(non_binary_bec_df, 'llama_3_1_8b_score')

In [17]:
# if running Llama 3 on hardware with GPU, you can re-install torch with GPU support using the following commands:

# !pip uninstall torch torchvision -y
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

### Review Findings

In [18]:
import pandas as pd

bec_df = pd.read_csv('bec_df_with_scores_20260502.csv')
non_binary_bec_df = pd.read_csv('non_binary_bec_df_with_scores_20260503.csv')

In [19]:
bec_df_ordered = bec_df[['Sentence', 'Sent_TM', 'Sent_AM', 'Sent_TAM', 'Template', 'Person', 'Gender', 'Profession', 'Prof_Gender', 'bert_score', 'gpt4_1_nano_score', 'gpt5_4_score', 'llama_3_1_8b_score']]
non_binary_bec_df_ordered = non_binary_bec_df[['Sentence', 'Sent_TM', 'Sent_AM', 'Sent_TAM', 'Template', 'Person', 'Gender', 'Profession', 'Prof_Gender', 'bert_score', 'gpt4_1_nano_score', 'gpt5_4_score', 'llama_3_1_8b_score']]
bec_combined_df = pd.concat([bec_df_ordered, non_binary_bec_df_ordered], ignore_index=True)

#### <b>Mean scores by Target Gender and Profession</b>
* Even for statistically balanced professions, each model shows bias with some professions, interestingly different models exhibit directional profession-level biases. GPT 5.4 seems to minimize some of the biases seen in GPT 4.1 Nano, overcorrecting in some cases.

* BERT is the only model with non-binary scores (more on this in the Limitations section), which have a few interesting subpatterns. In some cases, BERT shows similar negative bias toward the non-binary target pronouns as male or female pronouns, while in others the non-binary score lies between male and female, and still in other cases, BERT actually shows positive bias toward the non-binary pronoun relative to the others. This may be partly due to the model's low prior probabilities of choosing non-binary pronouns.

* All four models consistently (BERT less so) show bias toward the dominant gender in statistically unbalanced professions.


In [20]:
mean_scores =data_utils.get_profession_mean_scores(bec_combined_df)
mean_scores.head(51)

,Profession,Prof_Gender,Gender,bert_score,gpt4_1_nano_score,gpt5_4_score,llama_3_1_8b_score
0,bartender,balanced,Non-binary,0.384849,0.000000,0.000000,0.000000
1,bartender,balanced,female,-0.518063,0.307029,0.415012,0.971589
2,bartender,balanced,male,-0.294961,1.844322,0.822479,1.191833
3,billing clerk,female,Non-binary,-0.234076,0.000000,0.000000,0.000000
4,billing clerk,female,female,0.222025,1.333572,1.543402,2.612328
5,billing clerk,female,male,-0.306546,0.606222,-0.335164,-2.565012
6,bookkeeper,female,Non-binary,0.259809,0.000000,0.000000,0.000000
7,bookkeeper,female,female,-0.183140,0.895272,1.155071,1.995050
8,bookkeeper,female,male,-0.144452,0.118129,-0.601764,0.316678
9,bus mechanic,male,Non-binary,0.011697,0.000000,0.000000,0.000000


In [21]:
viz.plot_mean_profession_scores_for_model(mean_scores, 'bert_score', 'BERT', 'balanced')
viz.plot_mean_profession_scores_for_model(mean_scores, 'bert_score', 'BERT', 'male')
viz.plot_mean_profession_scores_for_model(mean_scores, 'bert_score', 'BERT', 'female')
viz.plot_mean_profession_scores_for_model(mean_scores, 'gpt5_4_score', 'GPT-5.4', 'balanced')
viz.plot_mean_profession_scores_for_model(mean_scores, 'gpt5_4_score', 'GPT-5.4', 'male')
viz.plot_mean_profession_scores_for_model(mean_scores, 'gpt5_4_score', 'GPT-5.4', 'female')
viz.plot_mean_profession_scores_for_model(mean_scores, 'gpt4_1_nano_score', 'GPT-4.1 Nano', 'balanced')
viz.plot_mean_profession_scores_for_model(mean_scores, 'gpt4_1_nano_score', 'GPT-4.1 Nano', 'male')
viz.plot_mean_profession_scores_for_model(mean_scores, 'gpt4_1_nano_score', 'GPT-4.1 Nano', 'female')
viz.plot_mean_profession_scores_for_model(mean_scores, 'llama_3_1_8b_score', 'Llama 3.1 8B Instruct AWQ INT4', 'balanced')
viz.plot_mean_profession_scores_for_model(mean_scores, 'llama_3_1_8b_score', 'Llama 3.1 8B Instruct AWQ INT4', 'male')
viz.plot_mean_profession_scores_for_model(mean_scores, 'llama_3_1_8b_score', 'Llama 3.1 8B Instruct AWQ INT4', 'female')

#### <b>Mean scores by Target Gender and Profession Gender</b>

* GPT 4.1 Nano shows the strongest bias toward male pronouns with balanced professions, but this difference appears largely mitigated in GPT 5.4. Llama 3.1 8B slightly favors female pronouns in these cases, while BERT slightly leans away from female pronouns.

* Similarly, GPT 4.1 Nano shows the weakest bias toward female pronouns in female-dominated professions, but the strongest toward male pronouns in male-dominated professions. In both cases, GPT 5.4 shows more skew toward female pronouns, but not as much as Llama 3.1 8B.

* BERT shows substntially less bias away from non-binary pronouns than male in female professions or female in male professions and less bias toward the dominant gender in unbalanced professions than later, larger models.

In [22]:
prof_gender_means = data_utils.get_prof_gender_mean_scores(bec_combined_df)
prof_gender_means

,Prof_Gender,Gender,Model,Score
0,balanced,Non-binary,Bert,-0.176159
1,balanced,female,Bert,-0.377158
2,balanced,male,Bert,-0.202824
3,female,Non-binary,Bert,-0.330485
4,female,female,Bert,0.420192
5,female,male,Bert,-1.306317
6,male,Non-binary,Bert,-0.092855
7,male,female,Bert,-1.145317
8,male,male,Bert,-0.107318
9,balanced,Non-binary,Gpt4 1 Nano,0.000000


In [24]:
viz.plot_mean_prof_gender_scores(prof_gender_means, 'balanced')
viz.plot_mean_prof_gender_scores(prof_gender_means, 'female')
viz.plot_mean_prof_gender_scores(prof_gender_means, 'male')
